# Web-Gold-40K — causal recovery v2

This notebook leaves `kaggle_gold.ipynb` and `kaggle_gold_recovery_v1.ipynb` unchanged. It audits/re-exports causal recovery transitions, runs a 16-row smoke test, then a controlled 5,000/500-row, 5-epoch experiment. Training never uses test examples; the dataset-only audit may count test labels but never produces test predictions.

In [ ]:
# 1. Pull modular code and record the exact environment.
from pathlib import Path
import importlib.metadata as metadata
import json, os, subprocess, sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
environment = {name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
Path('/kaggle/working/gold_recovery_v2_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))

In [ ]:
# 2. Locate the attached dataset; do not download or extract it.
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')
def find_split_root(root: Path) -> Path:
    if all((root / name).is_file() for name in SPLIT_FILES):
        return root
    candidates = []
    for current, _, files in os.walk(root, followlinks=True):
        if set(SPLIT_FILES).issubset(files):
            candidates.append(Path(current))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one structured split folder; found {candidates}')
    return candidates[0]
DATA_ROOT = find_split_root(ATTACHED_ROOT).resolve()
print('DATA_ROOT =', DATA_ROOT)
print('Dataset remains read-only under /kaggle/input.')

In [ ]:
# 3. Dataset-only full audit and causal manifest export (no image copies).
AUDIT_DIR = Path('/kaggle/working/recovery_transition_v2')
command = [sys.executable, 'scripts/build_recovery_transitions.py', '--data-root', str(DATA_ROOT), '--output-dir', str(AUDIT_DIR)]
audit_env = os.environ.copy()
audit_env['PYTHONPATH'] = str(SOURCE_ROOT) + os.pathsep + audit_env.get('PYTHONPATH', '')
subprocess.run(command, check=True, env=audit_env)
audit = json.loads((AUDIT_DIR / 'recovery_class_audit.json').read_text(encoding='utf-8'))
for split, details in audit['splits'].items():
    print(split, details['class_audit']['targeted_counts'], details['transition_report'])
print('This cell reads test labels only for dataset completeness; training below never loads test rows.')

In [ ]:
# 4. Choose one stage. Run smoke first, then restart and run mini.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
STAGE = 'smoke'  # 'smoke' or 'mini'
SEED, SMOKE_ROWS = 42, 16
MINI_TRAIN_ROWS, MINI_VAL_ROWS, MINI_EPOCHS = 5_000, 500, 5
assert STAGE in {'smoke', 'mini'}
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)
cfg = load_config('configs/backbones/qwen2vl_2b_gold_v2.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['num_workers'] = 0 if STAGE == 'smoke' else 4
assert cfg['data']['executed_action_post'] and cfg['data']['recovery_transitions']
assert cfg['backbone']['preserve_spatial_tokens'] and cfg['model']['spatial_grounding']
assert cfg['loss']['hierarchical_recovery'] and cfg['loss']['dynamic_weighting'] == 'uncertainty'
print('GPU:', torch.cuda.get_device_name(0), '| STAGE:', STAGE)

In [ ]:
# 5. Inspect causal streams before allocating the model.
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split
from web_agent.train.gold_stages import build_processor
processor = build_processor(cfg)
train_records = load_gold_split(cfg, 'train')
inspection_loader = build_gold_dataloader(cfg, 'train', processor, records=train_records, limit=16, batch_size=4, shuffle=False, num_workers=0, seed=SEED, smoke=True, trajectory_records=train_records)
inspection_batch = next(iter(inspection_loader))
assert {'pre_input_ids', 'post_input_ids', 'label_needs_recovery'}.issubset(inspection_batch)
assert 'input_ids' not in inspection_batch
pre_prompt = processor.decode(inspection_batch['pre_input_ids'][0], skip_special_tokens=False)
post_prompt = processor.decode(inspection_batch['post_input_ids'][0], skip_special_tokens=False)
assert 'executed_action:' not in pre_prompt
assert 'executed_action:' in post_prompt
if (inspection_batch['label_needs_recovery'] > 0).any():
    assert 'recovery_input_ids' in inspection_batch and 'recovery_row_indices' in inspection_batch
for key, value in inspection_batch.items():
    if torch.is_tensor(value):
        print(f'{key:30} {tuple(value.shape)}')

In [ ]:
# 6. Run smoke or the controlled five-epoch mini experiment.
from web_agent.train.gold_stages import run_gold_mini, run_gold_smoke
from web_agent.utils.results import save_mini_diagnostics_json, save_mini_result_csv
if STAGE == 'smoke':
    stage_report = run_gold_smoke(cfg, processor=processor, rows=SMOKE_ROWS, seed=SEED)
else:
    stage_report = run_gold_mini(cfg, processor=processor, train_rows=MINI_TRAIN_ROWS, val_rows=MINI_VAL_ROWS, epochs=MINI_EPOCHS, seed=SEED)
report_path = Path(f'/kaggle/working/gold_recovery_v2_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = diagnostics_path = None
if STAGE == 'mini':
    result_csv_path = save_mini_result_csv(stage_report, '/kaggle/working/gold_recovery_v2_result.csv')
    diagnostics_path = save_mini_diagnostics_json(stage_report, '/kaggle/working/gold_recovery_v2_diagnostics.json')
print(json.dumps(stage_report, indent=2))
print('Report:', report_path, '| CSV:', result_csv_path, '| diagnostics:', diagnostics_path)

In [ ]:
# 7. Enforce engineering gates; quality is reported, never hidden.
assert stage_report['status'] == 'PASS'
if STAGE == 'smoke':
    assert stage_report['processed_rows'] == SMOKE_ROWS and stage_report['parameter_changed']
    print('SMOKE PASSED. Restart the kernel, change STAGE to mini, then Run All.')
else:
    assert stage_report['train_rows'] == MINI_TRAIN_ROWS
    assert stage_report['val_rows'] == MINI_VAL_ROWS and stage_report['test_rows_read'] == 0
    assert len(stage_report['history']) == MINI_EPOCHS
    assert len(stage_report['epoch_checkpoints']) == MINI_EPOCHS
    assert stage_report['checkpoint_roundtrip'] and result_csv_path.is_file() and diagnostics_path.is_file()
    selected = stage_report['history'][stage_report['quality_gates']['selected_epoch']]
    print('MINI ENGINEERING PASS; full/headline training remains blocked.')
    print('Primary localization: mean IoU =', selected['bbox_mean_iou'], 'Recall@IoU50 =', selected['bbox_recall_iou50'])
    print('Needs-recovery macro-F1 =', selected['needs_recovery_macro_f1'])
    print('Attempted-strategy macro-F1 =', selected['strategy_attempted_macro_f1'])
    print('Transition recovery-outcome MCC =', selected['recovery_outcome_mcc'])

## Decision rule

Do not start full training merely because the notebook completed. Review the audit for missing `RETRY`, `ABORT`, `BACKTRACK`, and `LOOP_DETECTED`; complete the two-person human review; compare all five epochs using MCC/macro-F1 and bbox IoU/Recall@IoU50; and advance only if the predeclared gates improve without a material outcome/action regression.